# Lakekeeper Iceberg demo
Configure Spark to use the Lakekeeper REST catalog, write a tiny Iceberg table into MinIO, and read it back.

In [1]:
import os
from pyspark.sql import SparkSession, functions as F

catalog = "lk"
namespace = "demo"
table = "visits"
full_table = "lk.demo.visits"

access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab"))
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://127.0.0.1:9000")
hconf.set("fs.s3a.access.key", access_key)
hconf.set("fs.s3a.secret.key", secret_key)
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.connection.ssl.enabled", "false")

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {catalog}.{namespace}")


DataFrame[]

In [2]:
data = [
    ("2025-01-01", "alice", "web", 3),
    ("2025-01-01", "bob", "email", 1),
    ("2025-01-02", "alice", "ads", 2),
    ("2025-01-02", "carol", "web", 5),
]

df = (
    spark.createDataFrame(data, ["event_date", "user_id", "source", "visits"])
    .withColumn("ingested_at", F.current_timestamp())
)

df.writeTo(full_table).using("iceberg").createOrReplace()

spark.table(full_table).orderBy("event_date", "user_id").show(truncate=False)
spark.sql(f"SELECT event_date, SUM(visits) AS total_visits FROM {full_table} GROUP BY event_date ORDER BY event_date").show()


+----------+-------+------+------+--------------------------+
|event_date|user_id|source|visits|ingested_at               |
+----------+-------+------+------+--------------------------+
|2025-01-01|alice  |web   |3     |2025-11-24 18:53:52.278621|
|2025-01-01|bob    |email |1     |2025-11-24 18:53:52.278621|
|2025-01-02|alice  |ads   |2     |2025-11-24 18:53:52.278621|
|2025-01-02|carol  |web   |5     |2025-11-24 18:53:52.278621|
+----------+-------+------+------+--------------------------+

+----------+------------+
|event_date|total_visits|
+----------+------------+
|2025-01-01|           4|
|2025-01-02|           7|
+----------+------------+



In [3]:
location_row = (
    spark.sql(f"DESCRIBE TABLE EXTENDED {full_table}")
    .filter("col_name = 'Location'")
    .collect()
)
if location_row:
    print(f"Table data lives in MinIO at: {location_row[0].data_type}")

spark.read.format("iceberg").load(full_table).orderBy("ingested_at").show(truncate=False)


Table data lives in MinIO at: s3://mydatalab/warehouse/019ab737-1017-7393-ad93-58098fd29377/019ab737-153c-7a10-892e-fcc65b3febf8
+----------+-------+------+------+--------------------------+
|event_date|user_id|source|visits|ingested_at               |
+----------+-------+------+------+--------------------------+
|2025-01-01|alice  |web   |3     |2025-11-24 18:53:52.278621|
|2025-01-01|bob    |email |1     |2025-11-24 18:53:52.278621|
|2025-01-02|alice  |ads   |2     |2025-11-24 18:53:52.278621|
|2025-01-02|carol  |web   |5     |2025-11-24 18:53:52.278621|
+----------+-------+------+------+--------------------------+

